# IEE575 — Bayesian Optimization: Real-World Demo
## Tuning a PID Controller with a GP Surrogate

**Arizona State University — Applied Stochastic Operations Research Models**

---

> **This notebook is a class demo.** There are no submission tasks.
> Explore the cells, change parameters, and observe how BO behaves on a real engineering problem.


## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from scipy.stats import norm
from IPython.display import clear_output
import warnings
warnings.filterwarnings('ignore')

rng = np.random.default_rng(42)   # fixed seed — all randomness flows through here


---
## 1. The System: Mass–Spring–Damper

We want to move a mass from rest to a **target position** $r = 1$ m and hold it there.

The physical system is a **mass–spring–damper**:

$$m\ddot{x} + c\dot{x} + kx = u(t)$$

| Symbol | Meaning | Value |
|--------|---------|-------|
| $m$ | mass | 1 kg |
| $c$ | damping coefficient | 0.5 N·s/m |
| $k$ | spring constant | 1 N/m |
| $u(t)$ | control force (what we choose) | — |

### The PID controller

The controller computes a force $u(t)$ based on the **error** $e(t) = r - x(t)$:

$$u(t) = K_p\, e(t) + K_i \int_0^t e(\tau)\,d\tau + K_d\, \dot{e}(t)$$

- **$K_p$ (Proportional):** push harder the further away you are
- **$K_i$ (Integral):** correct for persistent steady-state error
- **$K_d$ (Derivative):** brake before you overshoot

**The tuning problem:** find $(K_p, K_i, K_d)$ that makes the system reach $r=1$ quickly,
without oscillating wildly, and without demanding excessive force.

This is genuinely hard. The three gains interact nonlinearly. Too much $K_p$ → oscillation.
Too much $K_d$ → sluggish. Wrong $K_i$ → drift. Practitioners have tuned these by hand for
decades. BO finds a good configuration in ~30 evaluations.


---
## 2. The Simulator (our black box)

Each call to `simulate_pid(Kp, Ki, Kd)` integrates the ODE for 10 seconds and returns
the full trajectory. We never look inside the function when optimizing — we only observe
the scalar cost.


In [ ]:
# System parameters
M, C_damp, K_spring = 1.0, 0.5, 1.0
T_END, DT = 10.0, 0.01
T_EVAL = np.arange(0, T_END + DT, DT)
R = 1.0   # setpoint (target position)

def simulate_pid(Kp, Ki, Kd):
    """
    Simulate the mass-spring-damper under PID control.

    Returns
    -------
    t  : np.ndarray  — time points
    x  : np.ndarray  — position trajectory
    u  : np.ndarray  — control force trajectory
    """
    integral = [0.0]
    u_log    = []

    def ode(t, y):
        x, x_dot = y
        e     = R - x
        e_dot = -x_dot          # derivative of error (r is constant)
        integral[0] += e * DT
        u = Kp * e + Ki * integral[0] + Kd * e_dot
        u_log.append(u)
        x_ddot = (u - C_damp * x_dot - K_spring * x) / M
        return [x_dot, x_ddot]

    sol = solve_ivp(ode, [0, T_END], [0.0, 0.0], t_eval=T_EVAL,
                    method='RK45', max_step=DT)
    return sol.t, sol.y[0], np.array(u_log[:len(sol.t)])


def cost(Kp, Ki, Kd, w_settle=1.0, w_overshoot=2.0, w_effort=0.1):
    """
    ISE-style cost: penalises settling time, overshoot, and control effort.
    Clips at 1e4 to handle unstable configurations gracefully.
    Lower is better.
    """
    try:
        t, x, u = simulate_pid(Kp, Ki, Kd)
        if np.any(np.abs(x) > 1e4):          # catch instability
            return 1e4
        settled  = np.where(np.abs(x - R) < 0.02)[0]
        t_settle = t[settled[0]] if len(settled) > 0 else T_END
        overshoot = max(0.0, np.max(x) - R)
        effort = np.trapezoid(u**2, t)        # np.trapz deprecated in NumPy 2.0
        return min(w_settle * t_settle + w_overshoot * overshoot + w_effort * effort, 1e4)
    except Exception:
        return 1e4


---
## 3. Manual Tuning — Why Is This Hard?

Before running BO, let's see what happens when we tune by hand.
Try a few combinations and observe:
- How quickly does the system reach $r = 1$?
- Does it overshoot? Oscillate?
- What does the control force look like?


In [ ]:
def plot_response(Kp, Ki, Kd, ax_pos=None, ax_u=None, label=None, color=None):
    t, x, u = simulate_pid(Kp, Ki, Kd)
    c_val    = cost(Kp, Ki, Kd)

    if ax_pos is None:
        fig, (ax_pos, ax_u) = plt.subplots(2, 1, figsize=(9, 5), sharex=True)

    lbl = label or f"Kp={Kp}, Ki={Ki}, Kd={Kd}  [cost={c_val:.2f}]"
    ax_pos.plot(t, x, label=lbl, color=color)
    ax_pos.axhline(R,    color='grey', linestyle='--', linewidth=1)
    ax_pos.axhline(R*1.02, color='grey', linestyle=':', linewidth=0.8)
    ax_pos.axhline(R*0.98, color='grey', linestyle=':', linewidth=0.8)
    ax_pos.set_ylabel("Position x (m)")

    ax_u.plot(t[:len(u)], u, color=color, alpha=0.7)
    ax_u.set_ylabel("Control force u (N)")
    ax_u.set_xlabel("Time (s)")

    return ax_pos, ax_u


# ── Three hand-tuned configurations ──────────────────────────────────────────
configs = [
    (1.0, 0.0, 0.0, "P only"),
    (5.0, 1.0, 0.5, "Reasonable PID"),
    (20.0, 5.0, 0.1, "Aggressive (oscillates)"),
]
colors = ['steelblue', 'seagreen', 'tomato']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
for (Kp, Ki, Kd, label), col in zip(configs, colors):
    plot_response(Kp, Ki, Kd, ax_pos=ax1, ax_u=ax2, label=label, color=col)

ax1.legend(fontsize=8)
ax1.set_title("Manual Tuning — Three Configurations")
ax1.set_ylim(-0.2, 2.0)
plt.tight_layout()
plt.show()

print("Costs:")
for Kp, Ki, Kd, label in configs:
    print(f"  {label:30s}  cost = {cost(Kp, Ki, Kd):.3f}")


**Discussion:**
- The P-only controller has steady-state error (spring pulls it back, no integral correction).
- The reasonable PID reaches the setpoint but still oscillates briefly.
- The aggressive configuration overshoots badly — high cost despite fast initial approach.

This is the search landscape BO has to navigate. It is **not convex**, and gradient information
is unavailable (we are treating the ODE simulator as a black box).


---
## 4. Bayesian Optimization Setup

**Search space:**

| Parameter | Range |
|-----------|-------|
| $K_p$ | [0.1, 30] |
| $K_i$ | [0.0, 10] |
| $K_d$ | [0.0,  5] |

We run 8 random initial evaluations, then 25 BO iterations.


In [ ]:
# ── Search space bounds ───────────────────────────────────────────────────────
# Ki is capped at 5 (not 10): large Ki with small Kd causes integrator wind-up,
# pushing costs to 80 000+ and making the GP landscape unlearnable.
PBOUNDS = np.array([
    [0.5, 15.0],   # Kp
    [0.0,  5.0],   # Ki  <- tightened from 10
    [0.0,  5.0],   # Kd
])

def normalize(X):
    """Scale each dimension to [0, 1] for the GP."""
    return (X - PBOUNDS[:, 0]) / (PBOUNDS[:, 1] - PBOUNDS[:, 0])

def denormalize(X_norm):
    return X_norm * (PBOUNDS[:, 1] - PBOUNDS[:, 0]) + PBOUNDS[:, 0]


# ── Acquisition: Expected Improvement ────────────────────────────────────────
def expected_improvement(X_cand, X_obs, gp, xi=0.01):
    mu, sigma  = gp.predict(X_cand, return_std=True)
    f_star     = gp.predict(X_obs).max()
    sigma      = np.maximum(sigma, 1e-9)
    Z          = (mu - f_star - xi) / sigma
    ei         = (mu - f_star - xi) * norm.cdf(Z) + sigma * norm.pdf(Z)
    ei[sigma < 1e-9] = 0.0
    return ei


# ── GP surrogate ─────────────────────────────────────────────────────────────
kernel = C(1.0, (1e-3, 1e3)) * RBF(length_scale=np.ones(3),
                                     length_scale_bounds=(1e-2, 10))
gp = GaussianProcessRegressor(
    kernel=kernel,
    n_restarts_optimizer=5,
    normalize_y=True
)


# ── Candidate grid (in normalized space) ──────────────────────────────────────
N_CAND = 8000
X_cand_norm = rng.uniform(0, 1, (N_CAND, 3))


---
## 5. Running BO

The loop prints a one-line summary per iteration so you can watch it converge in real time.
The full visualization comes after.


In [ ]:
# ── Shared initial random evaluations ────────────────────────────────────────
# These SAME points are used as the start for both BO and the random baseline.
# This isolates the surrogate as the only variable between the two strategies.
n_init = 5

X_init_norm = rng.uniform(0, 1, (n_init, 3))
X_init      = denormalize(X_init_norm)
Y_init_raw  = np.array([cost(*row) for row in X_init])

# Log-transform: compresses the 3-order-of-magnitude cost range,
# giving the GP a well-behaved surface to model.
Y_init_log  = np.log1p(Y_init_raw)

# Working copies for the BO run
X_obs_norm = X_init_norm.copy()
Y_obs_log  = Y_init_log.copy()
Y_obs_raw  = Y_init_raw.copy()

print("Shared initial evaluations (used by both BO and random baseline):")
for i, (x, y) in enumerate(zip(X_init, Y_init_raw)):
    print(f"  [{i+1:2d}] Kp={x[0]:5.2f}  Ki={x[1]:5.2f}  Kd={x[2]:5.2f}  ->  cost={y:.3f}")
print(f"\nBest so far: {Y_obs_raw.min():.3f}")


In [ ]:
# ── BO loop ──────────────────────────────────────────────────────────────────
n_bo_iter = 35
# bo_conv: best raw cost at every evaluation (init phase expanded point-by-point)
bo_conv   = [float(Y_init_raw[:k+1].min()) for k in range(n_init)]

print(f"{"Iter":>4}  {"Kp":>6}  {"Ki":>6}  {"Kd":>6}  {"cost":>8}  {"best":>8}")
print("-" * 50)

for i in range(n_bo_iter):
    # Fit surrogate on negated log-cost (GP maximises; we minimise cost)
    gp.fit(X_obs_norm, -Y_obs_log)

    # Find next candidate via EI
    ei          = expected_improvement(X_cand_norm, X_obs_norm, gp)
    x_next_norm = X_cand_norm[np.argmax(ei)]
    x_next      = denormalize(x_next_norm)

    # Evaluate true cost and log-transform for GP
    y_next_raw  = cost(*x_next)
    y_next_log  = np.log1p(y_next_raw)

    # Update dataset
    X_obs_norm = np.vstack([X_obs_norm, x_next_norm])
    Y_obs_log  = np.append(Y_obs_log, y_next_log)
    Y_obs_raw  = np.append(Y_obs_raw, y_next_raw)
    bo_conv.append(float(Y_obs_raw.min()))

    print(f"{i+1:4d}  {x_next[0]:6.2f}  {x_next[1]:6.2f}  {x_next[2]:6.2f}  "
          f"{y_next_raw:8.3f}  {Y_obs_raw.min():8.3f}")

best_idx    = np.argmin(Y_obs_raw)
best_params = denormalize(X_obs_norm[best_idx])
print(f"\nBO best:  Kp={best_params[0]:.3f}  Ki={best_params[1]:.3f}  Kd={best_params[2]:.3f}")
print(f"  Cost: {Y_obs_raw.min():.4f}")

# ── Random baseline: same init, then purely random continuation ───────────────
rng_rand     = np.random.default_rng(1042)   # independent seed
X_rand_extra = denormalize(rng_rand.uniform(0, 1, (n_bo_iter, 3)))
Y_rand_extra = np.array([cost(*row) for row in X_rand_extra])
Y_rand_all   = np.concatenate([Y_init_raw, Y_rand_extra])
rand_conv    = [float(Y_rand_all[:k+1].min()) for k in range(n_init + n_bo_iter)]

print(f"\nRandom baseline best (same budget, same start): {Y_rand_all.min():.4f}")


---
## 6. Results


In [ ]:
# ── Convergence: BO vs random search ─────────────────────────────────────────
n_total   = n_init + n_bo_iter
hand_cost = cost(5.0, 1.0, 0.5)

fig, ax = plt.subplots(figsize=(9, 4))

ax.plot(range(1, n_total + 1), rand_conv,
        color='#f4a582', lw=2, linestyle='--', marker='s', markersize=3.5,
        label='Random search — shared init + random continuation')

ax.plot(range(1, n_total + 1), bo_conv,
        color='#2166ac', lw=2.2, marker='o', markersize=3.5,
        label='Bayesian Optimization — shared init + GP-guided')

# Shade the shared init phase
ax.axvspan(1, n_init, alpha=0.10, color='grey', label='Shared init (identical for both)')
ax.axvline(n_init, color='grey', lw=1.2, linestyle=':')
ax.text(n_init + 0.5, max(rand_conv) * 0.92,
        'BO takes over ->', fontsize=8, color='#2166ac', va='top')

ax.axhline(hand_cost, color='#555', linestyle=':', lw=1.3,
           label=f'Hand-tuned = {hand_cost:.1f}')

ax.set_xlabel("Number of simulator evaluations", fontsize=11)
ax.set_ylabel("Best cost found so far", fontsize=11)
ax.set_title("BO vs. Random Search -- PID Tuning\n"
             "Same starting point, same budget. Only the strategy differs.", fontsize=12)
ax.legend(fontsize=9)
ax.grid(True, alpha=0.35)
plt.tight_layout()
plt.show()

print(f"Shared init best : {Y_init_raw.min():.3f}")
print(f"BO final best    : {Y_obs_raw.min():.3f}")
print(f"Random final best: {Y_rand_all.min():.3f}")
print(f"Hand-tuned       : {hand_cost:.3f}")
print(f"BO vs random     : {100*(Y_rand_all.min()-Y_obs_raw.min())/Y_rand_all.min():.1f}% improvement")


In [ ]:
# ── Compare: best BO vs hand-tuned ──────────────────────────────────────────
best_manual = (5.0, 1.0, 0.5)   # 'Reasonable PID' from Section 3

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

plot_response(*best_manual, ax_pos=ax1, ax_u=ax2,
              label=f'Hand-tuned  (cost={cost(*best_manual):.2f})', color='tomato')
plot_response(*best_params, ax_pos=ax1, ax_u=ax2,
              label=f'BO result   (cost={Y_obs_raw.min():.2f})', color='steelblue')

ax1.set_title("Hand-Tuned vs. BO-Tuned PID")
ax1.legend(fontsize=9)
ax1.set_ylim(-0.1, 1.6)
plt.tight_layout()
plt.show()

print(f"Hand-tuned cost : {cost(*best_manual):.3f}")
print(f"BO-tuned cost   : {Y_obs_raw.min():.3f}")
print(f"Improvement     : {100*(cost(*best_manual) - Y_obs_raw.min())/cost(*best_manual):.1f}%")


In [ ]:
# ── Learned length-scales: which gain matters most? ─────────────────────────
ls = gp.kernel_.k2.length_scale   # after fitting, sklearn stores optimized params here
param_names = ['Kp', 'Ki', 'Kd']

fig, ax = plt.subplots(figsize=(5, 3))
bars = ax.bar(param_names, ls, color=['steelblue', 'seagreen', 'tomato'])
ax.set_ylabel("Learned length-scale")
ax.set_title("GP Length-Scales per PID Gain
(larger = less sensitive along that axis)")
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, ls):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.05,
            f"{val:.2f}", ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print("\nInterpretation:")
most_sensitive = param_names[np.argmin(ls)]
least_sensitive = param_names[np.argmax(ls)]
print(f"  Most sensitive gain  (shortest ℓ): {most_sensitive}")
print(f"  Least sensitive gain (longest  ℓ): {least_sensitive}")


### What the length-scales tell us

The GP automatically learns how sensitive the cost is to each gain.
A **short length-scale** means the cost changes rapidly along that axis —
small changes to that gain have a big effect. A **long length-scale** means
the cost is relatively flat along that axis — the optimizer is less concerned about precision there.

This is **automatic relevance determination (ARD)** — the same idea that motivates
SAASBO in high dimensions. The GP discovers structure in the problem without being told.


---
## 7. Explore Yourself

Try changing any of the parameters below and re-running the BO loop.

**Questions to explore:**
- What happens with only 3 random initial points instead of 8?
- What if you increase `w_overshoot` to 5.0? Does BO find a less oscillatory solution?
- What if you widen the $K_d$ range to [0, 20]? Does the search space change the result?
- Replace the RBF kernel with Matérn-1/2. How does convergence change?


In [ ]:
# ── Sandbox: re-run with custom settings ─────────────────────────────────────
# Modify any of these and re-run from here

N_INIT_SANDBOX = 5       # number of shared random initial evaluations
N_BO_SANDBOX   = 35      # number of BO iterations
W_SETTLE       = 1.0     # weight on settling time
W_OVERSHOOT    = 2.0     # weight on overshoot (try 5.0)
W_EFFORT       = 0.1     # weight on control effort

PBOUNDS_SANDBOX = np.array([
    [0.5, 15.0],   # Kp
    [0.0,  5.0],   # Ki  <- try [0, 10] to see instability
    [0.0,  5.0],   # Kd
])

def cost_sandbox(Kp, Ki, Kd):
    return cost(Kp, Ki, Kd, w_settle=W_SETTLE, w_overshoot=W_OVERSHOOT, w_effort=W_EFFORT)

rng_s = np.random.default_rng(99)   # separate seed

# Shared init
X_s_norm = rng_s.uniform(0, 1, (N_INIT_SANDBOX, 3))
X_s      = X_s_norm * (PBOUNDS_SANDBOX[:,1] - PBOUNDS_SANDBOX[:,0]) + PBOUNDS_SANDBOX[:,0]
Y_s_raw  = np.array([cost_sandbox(*row) for row in X_s])
Y_s_log  = np.log1p(Y_s_raw)
bo_s_conv = [float(Y_s_raw[:k+1].min()) for k in range(N_INIT_SANDBOX)]

# Random baseline from same init
rng_rs      = np.random.default_rng(199)
X_rs_extra  = rng_rs.uniform(0, 1, (N_BO_SANDBOX, 3))
X_rs_phys   = X_rs_extra * (PBOUNDS_SANDBOX[:,1] - PBOUNDS_SANDBOX[:,0]) + PBOUNDS_SANDBOX[:,0]
Y_rs_extra  = np.array([cost_sandbox(*row) for row in X_rs_phys])
Y_rs_all    = np.concatenate([Y_s_raw, Y_rs_extra])
rand_s_conv = [float(Y_rs_all[:k+1].min()) for k in range(N_INIT_SANDBOX + N_BO_SANDBOX)]

# BO run
# <- Try swapping to Matern(nu=0.5) to see the bad-kernel effect from Lab 3
kernel_s = C(1.0) * RBF(length_scale=np.ones(3))
gp_s     = GaussianProcessRegressor(kernel=kernel_s, n_restarts_optimizer=3, normalize_y=True)
cand_s   = rng_s.uniform(0, 1, (N_CAND, 3))

X_s_obs  = X_s_norm.copy()
Y_s_obs  = Y_s_log.copy()
Y_s_raw2 = Y_s_raw.copy()

for i in range(N_BO_SANDBOX):
    gp_s.fit(X_s_obs, -Y_s_obs)
    ei_s    = expected_improvement(cand_s, X_s_obs, gp_s)
    xn_norm = cand_s[np.argmax(ei_s)]
    xn      = xn_norm * (PBOUNDS_SANDBOX[:,1] - PBOUNDS_SANDBOX[:,0]) + PBOUNDS_SANDBOX[:,0]
    yn_raw  = cost_sandbox(*xn)
    yn_log  = np.log1p(yn_raw)
    X_s_obs  = np.vstack([X_s_obs, xn_norm])
    Y_s_obs  = np.append(Y_s_obs, yn_log)
    Y_s_raw2 = np.append(Y_s_raw2, yn_raw)
    bo_s_conv.append(float(Y_s_raw2.min()))

best_s_norm = X_s_obs[np.argmin(Y_s_raw2)]
best_s_phys = best_s_norm * (PBOUNDS_SANDBOX[:,1] - PBOUNDS_SANDBOX[:,0]) + PBOUNDS_SANDBOX[:,0]

n_s_total = N_INIT_SANDBOX + N_BO_SANDBOX
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(range(1, n_s_total+1), rand_s_conv,
             color='#f4a582', lw=2, linestyle='--', marker='s', markersize=3, label='Random')
axes[0].plot(range(1, n_s_total+1), bo_s_conv,
             color='darkorange', lw=2, marker='o', markersize=3, label='BO')
axes[0].axvspan(1, N_INIT_SANDBOX, alpha=0.10, color='grey')
axes[0].axvline(N_INIT_SANDBOX, color='grey', lw=1, linestyle=':')
axes[0].set_xlabel("Evaluations"); axes[0].set_ylabel("Best cost")
axes[0].set_title("Sandbox Convergence"); axes[0].legend(); axes[0].grid(True, alpha=0.4)

t, x, u = simulate_pid(*best_s_phys)
axes[1].plot(t, x, color='darkorange',
             label=f'Kp={best_s_phys[0]:.2f} Ki={best_s_phys[1]:.2f} Kd={best_s_phys[2]:.2f}')
axes[1].axhline(R, color='grey', linestyle='--')
axes[1].set_xlabel("Time (s)"); axes[1].set_ylabel("Position (m)")
axes[1].set_title(f"Sandbox Best Response  [cost={Y_s_raw2.min():.3f}]")
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()
